In [1]:
import sys
from pathlib import Path

# Find the main project folder
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.preprocess import clean_text, create_shingles
from src.minhash import create_minhash
from src.jaccard import jaccard_similarity

# Two reviews to compare
review1 = "This product works really well"
review2 = "This product works very well"

# Clean and create shingles
shingles1 = create_shingles(clean_text(review1))
shingles2 = create_shingles(clean_text(review2))

# Calculate exact similarity
exact = jaccard_similarity(shingles1, shingles2)

# Create MinHash signatures
minhash1 = create_minhash(shingles1)
minhash2 = create_minhash(shingles2)

# Estimate similarity using MinHash
estimated = minhash1.jaccard(minhash2)

print("Exact Jaccard:", round(exact, 3))
print("MinHash estimate:", round(estimated, 3))

Exact Jaccard: 0.333
MinHash estimate: 0.352


In [2]:
# Compare MinHash accuracy using different numbers of permutations
# More permutations may improve accuracy but require more memory and time

permutation_counts = [64, 128, 256]

for num_perm in permutation_counts:

    # Create signatures using the current permutation count
    minhash1 = create_minhash(shingles1, num_perm)
    minhash2 = create_minhash(shingles2, num_perm)

    # Estimate similarity
    estimated = minhash1.jaccard(minhash2)

    # Calculate how far the estimate is from exact Jaccard
    error = abs(exact - estimated)

    print(f"Permutations: {num_perm}")
    print(f"Estimated Jaccard: {estimated:.3f}")
    print(f"Absolute error: {error:.3f}")
    print()

Permutations: 64
Estimated Jaccard: 0.344
Absolute error: 0.010

Permutations: 128
Estimated Jaccard: 0.352
Absolute error: 0.018

Permutations: 256
Estimated Jaccard: 0.332
Absolute error: 0.001



In [3]:
# Load the same reviews used for the exact Jaccard baseline
# This lets us compare MinHash against the correct results

import pandas as pd

file_path = project_root / "results" / "jaccard_subset_1000.csv"

reviews = pd.read_csv(file_path)

print("Reviews loaded:", len(reviews))
display(reviews.head())

Reviews loaded: 1000


,review_id,clean_text
0,0,this spray is really nice it smells really goo...
1,1,this product does what i need it to do i just ...
2,2,smells good feels great
3,3,felt synthetic
4,4,love it


In [5]:
# Create 2-word shingles for each review
# We use the same shingle size as the Jaccard baseline

reviews["shingles"] = reviews["clean_text"].apply(
    lambda text: create_shingles(text, k=2)
)

print("Shingles created for", len(reviews), "reviews")

# Display the first review's shingles
print(reviews["shingles"].iloc[0])

Shingles created for 1000 reviews
{('with', 'this'), ('try', 'it'), ('m', 'gonna'), ('medium', 'thickness'), ('it', 'smells'), ('and', 'does'), ('i', 'will'), ('so', 'i'), ('it', 'though'), ('smells', 'really'), ('really', 'good'), ('though', 'to'), ('goes', 'on'), ('like', 'you'), ('other', 'brands'), ('of', 'it'), ('is', 'really'), ('i', 'want'), ('to', 'other'), ('good', 'goes'), ('comparing', 'to'), ('with', 'yucky'), ('yucky', 'chemicals'), ('to', 'get'), ('need', 'a'), ('stick', 'with'), ('really', 'fine'), ('this', 'try'), ('want', 'i'), ('brands', 'with'), ('fine', 'and'), ('texture', 'i'), ('chemicals', 'so'), ('nice', 'it'), ('i', 'have'), ('i', 'am'), ('it', 'feels'), ('trick', 'i'), ('feels', 'like'), ('have', 'a'), ('thickness', 'i'), ('the', 'trick'), ('on', 'really'), ('say', 'it'), ('hair', 'medium'), ('this', 'spray'), ('a', 'lot'), ('really', 'nice'), ('gonna', 'stick'), ('will', 'say'), ('get', 'the'), ('am', 'comparing'), ('spray', 'is'), ('lot', 'of'), ('i', 'm'), 

In [6]:
# Create a MinHash signature for every review
# Each signature gives us a smaller representation of its shingles

reviews["minhash"] = reviews["shingles"].apply(
    lambda shingles: create_minhash(shingles, num_perm=128)
)

print("MinHash signatures created:", len(reviews))

MinHash signatures created: 1000


In [7]:
# Load the exact Jaccard pairs found by Roquibat
exact_pairs = pd.read_csv(
    project_root / "results" / "jaccard_pairs_1000.csv"
)

print("Exact similar pairs:", len(exact_pairs))
display(exact_pairs.head())

Exact similar pairs: 38


,review_id_1,review_id_2,similarity
0,4,415,1.000000
1,4,619,1.000000
2,38,116,1.000000
3,163,485,1.000000
4,241,243,0.803468


In [8]:
# Use the original review IDs to find the correct reviews
# This makes sure our MinHash results match Roquibat's baseline

reviews = reviews.set_index("review_id")

print("Reviews available:", len(reviews))
print("First few review IDs:", reviews.index[:5].tolist())

Reviews available: 1000
First few review IDs: [0, 1, 2, 3, 4]


In [9]:
# Compare MinHash estimates with the exact Jaccard scores
# for the 38 similar review pairs found by Roquibat

comparison_results = []

for _, pair in exact_pairs.iterrows():

    # Get the original review IDs
    id1 = int(pair["review_id_1"])
    id2 = int(pair["review_id_2"])

    # Get the MinHash signatures for both reviews
    minhash1 = reviews.loc[id1, "minhash"]
    minhash2 = reviews.loc[id2, "minhash"]

    # Calculate estimated similarity
    estimated = minhash1.jaccard(minhash2)

    # Get exact similarity from Roquibat's results
    exact = pair["similarity"]

    # Calculate the difference
    error = abs(exact - estimated)

    comparison_results.append({
        "review_id_1": id1,
        "review_id_2": id2,
        "exact_jaccard": exact,
        "minhash_estimate": estimated,
        "absolute_error": error
    })

# Convert results to a DataFrame
comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.head(10))

print("Pairs compared:", len(comparison_df))
print("Average absolute error:",
      round(comparison_df["absolute_error"].mean(), 4))

,review_id_1,review_id_2,exact_jaccard,minhash_estimate,absolute_error
0,4,415,1.000000,1.000000,0.000000
1,4,619,1.000000,1.000000,0.000000
2,38,116,1.000000,1.000000,0.000000
3,163,485,1.000000,1.000000,0.000000
4,241,243,0.803468,0.789062,0.014406
5,254,256,0.813253,0.812500,0.000753
6,374,375,1.000000,1.000000,0.000000
7,415,619,1.000000,1.000000,0.000000
8,417,659,1.000000,1.000000,0.000000
9,539,937,1.000000,1.000000,0.000000


Pairs compared: 38
Average absolute error: 0.0089


In [10]:
# Test how permutation count affects MinHash accuracy
permutation_counts = [64, 128, 256]

results = []

for num_perm in permutation_counts:

    # Create new MinHash signatures for this permutation count
    signatures = {}

    for review_id, row in reviews.iterrows():
        signatures[review_id] = create_minhash(
            row["shingles"],
            num_perm=num_perm
        )

    errors = []

    # Compare the same 38 review pairs
    for _, pair in exact_pairs.iterrows():
        id1 = int(pair["review_id_1"])
        id2 = int(pair["review_id_2"])

        estimate = signatures[id1].jaccard(signatures[id2])
        exact_score = pair["similarity"]

        errors.append(abs(exact_score - estimate))

    average_error = sum(errors) / len(errors)

    results.append({
        "permutations": num_perm,
        "average_error": average_error
    })

results_df = pd.DataFrame(results)

display(results_df)

,permutations,average_error
0,64,0.019873
1,128,0.008940
2,256,0.007107


In [11]:
# Save MinHash accuracy results for our research paper
results_df.to_csv(
    project_root / "results" / "minhash_accuracy.csv",
    index=False
)

print("MinHash accuracy results saved!")

MinHash accuracy results saved!
